# 归一化与动作表示：复现错配，检查分布与旋转

目标：区分统计量错配、字段维度错配和动作表示转换错误。输入为此前保存的官方／训练统计、原变换源码和原动作接口；输出为误差、可预期的失败及真实统计差异。此前调查未发现明显归一化尺度错误，因此本篇保留这一排除结果，不编造一次不存在的“归一化修复提升”。

## 使用方式与证据层次

每节先复现一个小型、可解释的错误，再展开真实源码、定位机制，最后核对历史设备记录与修复后的守卫。人工构造的参数树、状态探针和动作块只用于验证机制，不替代真实模型或成功率。默认不启动训练、模型服务或完整仿真。

真实模型检查须先选择已有匹配环境，设置 `AMD_DIAGNOSTICS_LIVE=1`，指定本地源码、配置、权重和独立输出目录；不下载资源，也不改原始实验目录。视频为已有完整回合，播放器直接内嵌数据。

In [1]:
from __future__ import annotations
from pathlib import Path
import json, os, sys, hashlib, dataclasses, tempfile, copy, time
import numpy as np
from IPython.display import display, Video
TOPIC = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "code/amd_benchmarks").is_dir())
EVIDENCE = TOPIC / "notebooks/results"
SNAPSHOTS = TOPIC / "code/diagnostics_sources"
DIAG = EVIDENCE / "diagnostics"
RUN_LIVE_CHECKS = os.environ.get("AMD_DIAGNOSTICS_LIVE", "0") == "1"
for row in json.loads((DIAG / "source-manifest.json").read_text()):
    assert hashlib.sha256((TOPIC / row["file"]).read_bytes()).hexdigest() == row["sha256"]
print("源码与证据指纹通过；默认运行小型复现和历史记录复算，真实模型检查：", RUN_LIVE_CHECKS)


源码与证据指纹通过；默认运行小型复现和历史记录复算，真实模型检查： False


## 1. 用真实统计量看清输入契约

状态为23维，模型动作是22维：位置3维、旋转向量3维、手部16维。仿真接口使用23维动作，因为旋转用4维四元数表示。两个23维数组语义不同，不能凭维数相同互换。

In [2]:
official = json.loads((DIAG / "official_norm_stats.json").read_text())["norm_stats"]
trained = json.loads((DIAG / "training_norm_stats.json").read_text())["norm_stats"]
for label, stats in (("官方", official), ("训练", trained)):
    print(label, {k: len(v["mean"]) for k,v in stats.items()})
assert len(official["state"]["mean"]) == 23
assert len(official["actions"]["mean"]) == 22


官方 {'state': 23, 'actions': 22}
训练 {'state': 23, 'actions': 22}


## 2. 展开实际分位数正变换与逆变换

以下函数直接来自原源码，保留小跨度通道的中心化分支与 `1e-6` 稳定项。先核对本次模型配置选择的是分位数还是均值／标准差方式，再使用对应函数。这里测试分位数路径；传入同一份统计量时应恢复原动作。

In [3]:
def _normalize_quantile(self, x, stats: NormStats):
    assert stats.q01 is not None
    assert stats.q99 is not None
    q01, q99 = stats.q01[..., : x.shape[-1]], stats.q99[..., : x.shape[-1]]
    ignore_mask = q99 - q01 < 1e-4
    center = (q99 + q01) / 2.0

    scaled = (x - q01) / (q99 - q01 + 1e-6) * 2.0 - 1.0
    centered = x - center
    return np.where(ignore_mask, centered, scaled)


def _unnormalize_quantile(self, x, stats: NormStats):
    assert stats.q01 is not None
    assert stats.q99 is not None
    q01, q99 = stats.q01, stats.q99
    center = (q99 + q01) / 2.0
    ignore_mask = q99 - q01 < 1e-4
    dim = q01.shape[-1]

    x_main = x[..., :dim]
    unscaled = (x_main + 1.0) / 2.0 * (q99 - q01 + 1e-6) + q01
    uncentered = x_main + center
    restored = np.where(ignore_mask, uncentered, unscaled)

    if dim < x.shape[-1]:
        return np.concatenate([restored, x[..., dim:]], axis=-1)
    return restored


In [4]:
from types import SimpleNamespace
stats = SimpleNamespace(**{k: np.asarray(v,dtype=np.float64) for k,v in official["actions"].items()})
raw = np.asarray(official["actions"]["mean"],dtype=np.float64)
normalized = _normalize_quantile(None, raw, stats)
restored = _unnormalize_quantile(None, normalized, stats)
np.testing.assert_allclose(restored, raw, atol=1e-10)
print("同统计量往返最大误差：", float(np.max(np.abs(restored-raw))))


同统计量往返最大误差： 1.1102230246251565e-16


## 3. 故障复现与修复验证：维度正确，统计量语义错误

把动作统计通道错移一位，仍是22维，正变换也不会报维度错；但部署端若继续使用原统计量，动作已变。这个人为故障只用于展示检查方法；不是此前设备故障的实测根因。

In [5]:
wrong_stats = SimpleNamespace(**{k:np.roll(v,1) for k,v in vars(stats).items()})
bad_normalized = _normalize_quantile(None, raw, wrong_stats)
bad_restored = _unnormalize_quantile(None, bad_normalized, stats)
error = float(np.max(np.abs(bad_restored-raw)))
assert error > 1e-5
print("错移通道但维度正确时的动作误差：",error)
repaired = _unnormalize_quantile(None, _normalize_quantile(None,raw,stats),stats)
np.testing.assert_allclose(repaired,raw,atol=1e-10)
# Record the exact statistics bytes used by a deployed checkpoint.
print("官方统计指纹：",hashlib.sha256((DIAG / "official_norm_stats.json").read_bytes()).hexdigest())


错移通道但维度正确时的动作误差： 44.59158983261078
官方统计指纹： 82b6a71f7ad170c9aa8e3d90e525f09304fb0c9dc7cb9d4467718a82c31e6111


## 4. 定位真实数据：官方与本次训练统计差多少

分别比较两个端点分位数，不把维度相同直接当成分布相同。原调查最大端点差约0.00318，未发现明显尺度错配；这只能排查此项，还要继续检查动作语义和时间对齐。

In [6]:
differences = {}
for field in ("state","actions"):
    for key in ("q01","q99"):
        a,b = np.asarray(official[field][key]),np.asarray(trained[field][key])
        assert a.shape==b.shape and np.isfinite(a).all() and np.isfinite(b).all()
        differences[field+"/"+key] = float(np.max(np.abs(a-b)))
print(differences)
print("最大端点差：",max(differences.values()))


{'state/q01': 0.0017205912247300198, 'state/q99': 0.002308586549758851, 'actions/q01': 0.003176609780639411, 'actions/q99': 0.0018323733501137696}
最大端点差： 0.003176609780639411


### 部署中的真实顺序：从权重目录读取统计资产

以下是原策略构造函数，保留“输入变换→归一化→模型→动作逆归一化”的顺序。未显式传入统计时，它从所选权重的资产目录加载，避免训练与部署各用一套统计。这里只保留源码定义；真实加载由加载篇的显式检查入口执行。


In [7]:
if False:  # Reference implementation; heavy imports belong to the matching live environment.
    def create_trained_policy(
        train_config: _config.TrainConfig,
        checkpoint_dir: pathlib.Path | str,
        *,
        repack_transforms: transforms.Group | None = None,
        sample_kwargs: dict[str, Any] | None = None,
        default_prompt: str | None = None,
        norm_stats: dict[str, transforms.NormStats] | None = None,
        pytorch_device: str | None = None,
    ) -> _policy.Policy:
        """Create a policy from a trained checkpoint.

        Args:
            train_config: The training config to use to create the model.
            checkpoint_dir: The directory to load the model from.
            repack_transforms: Optional transforms that will be applied before any other transforms.
            sample_kwargs: The kwargs to pass to the `sample_actions` method. If not provided, the default
                kwargs will be used.
            default_prompt: The default prompt to use for the policy. Will inject the prompt into the input
                data if it doesn't already exist.
            norm_stats: The norm stats to use for the policy. If not provided, the norm stats will be loaded
                from the checkpoint directory.
            pytorch_device: Device to use for PyTorch models (e.g., "cpu", "cuda", "cuda:0").
                          If None and is_pytorch=True, will use "cuda" if available, otherwise "cpu".

        Note:
            The function automatically detects whether the model is PyTorch-based by checking for the
            presence of "model.safensors" in the checkpoint directory.
        """
        repack_transforms = repack_transforms or transforms.Group()
        checkpoint_dir = download.maybe_download(str(checkpoint_dir))

        # Check if this is a PyTorch model by looking for model.safetensors
        weight_path = os.path.join(checkpoint_dir, "model.safetensors")
        is_pytorch = os.path.exists(weight_path)

        logging.info("Loading model...")
        if is_pytorch:
            model = train_config.model.load_pytorch(train_config, weight_path)
            model.paligemma_with_expert.to_bfloat16_for_selected_params("bfloat16")
        else:
            model = train_config.model.load(_model.restore_params(checkpoint_dir / "params", dtype=jnp.bfloat16))
        data_config = train_config.data.create(train_config.assets_dirs, train_config.model)
        if norm_stats is None:
            # We are loading the norm stats from the checkpoint instead of the config assets dir to make sure
            # that the policy is using the same normalization stats as the original training process.
            if data_config.asset_id is None:
                raise ValueError("Asset id is required to load norm stats.")
            norm_stats = _checkpoints.load_norm_stats(checkpoint_dir / "assets", data_config.asset_id)

        # Determine the device to use for PyTorch models
        if is_pytorch and pytorch_device is None:
            try:
                import torch

                pytorch_device = "cuda" if torch.cuda.is_available() else "cpu"
            except ImportError:
                pytorch_device = "cpu"

        return _policy.Policy(
            model,
            transforms=[
                *repack_transforms.inputs,
                transforms.InjectDefaultPrompt(default_prompt),
                *data_config.data_transforms.inputs,
                transforms.Normalize(norm_stats, use_quantiles=data_config.use_quantile_norm),
                *data_config.model_transforms.inputs,
            ],
            output_transforms=[
                *data_config.model_transforms.outputs,
                transforms.Unnormalize(norm_stats, use_quantiles=data_config.use_quantile_norm),
                *data_config.data_transforms.outputs,
                *repack_transforms.outputs,
            ],
            sample_kwargs=sample_kwargs,
            metadata=train_config.policy_metadata,
            is_pytorch=is_pytorch,
            pytorch_device=pytorch_device if is_pytorch else None,
        )



## 5. 原动作适配源码：22维旋转向量转23维四元数

下面展开原接口。只提取纯动作转换方法，在当前处理器环境执行。兼容层仅处理本地旧版旋转库的四元数参数接口，不改原设备的源码。验证应同时检查位置、手部关节和旋转矩阵，不能直接比较旋转向量与四元数数组。

In [8]:
from scipy.spatial.transform import Rotation
class QuatResult:
    def __init__(self, value): self.value=value
    def as_quat(self,scalar_first=False):
        q=self.value.as_quat()
        return np.r_[q[3],q[:3]] if scalar_first else q
class RotationCompatibility:
    @staticmethod
    def from_rotvec(value): return QuatResult(Rotation.from_rotvec(value))
R = RotationCompatibility
def _process_action(self, action: np.ndarray) -> np.ndarray:
    """Convert OpenPI policy actions to DexJoCo environment action format."""
    if self.dual_arm:
        r_xyz = action[:3]
        r_rotvec = action[3:6]
        r_hand = action[6:22]
        l_xyz = action[22:25]
        l_rotvec = action[25:28]
        l_hand = action[28:44]

        r_quat = R.from_rotvec(r_rotvec).as_quat(scalar_first=True)
        l_quat = R.from_rotvec(l_rotvec).as_quat(scalar_first=True)

        return np.concatenate([r_xyz, r_quat, l_xyz, l_quat, r_hand, l_hand])
    else:
        xyz = action[:3]
        rotvec = action[3:6]
        hand = action[6:22]

        quat = R.from_rotvec(rotvec).as_quat(scalar_first=True)

        return np.concatenate([xyz, quat, hand])


Probe = type("Probe",(),{"_process_action":_process_action})
probe = Probe(); probe.dual_arm=False


In [9]:
rotation_errors=[]
for key in ("mean","q01","q99"):
    action=np.asarray(official["actions"][key])
    converted=probe._process_action(action)
    assert converted.shape==(23,)
    np.testing.assert_array_equal(converted[:3],action[:3])
    np.testing.assert_array_equal(converted[7:],action[6:22])
    scalar_last=np.r_[converted[4:7],converted[3]]
    error=float(np.max(np.abs(Rotation.from_rotvec(action[3:6]).as_matrix()-Rotation.from_quat(scalar_last).as_matrix())))
    assert error<1e-12
    rotation_errors.append(error)
print("三种真实统计动作的旋转等价误差：",rotation_errors)
historical=json.loads((EVIDENCE / "system/strict_loader_action_interface_audit.json").read_text())
print("历史接口检查：",historical)


三种真实统计动作的旋转等价误差： [3.3306690738754696e-16, 2.220446049250313e-16, 0.0]
历史接口检查： {'scope': 'Source-level CPU action layout check; not policy inference, simulation or success evaluation', 'local_scipy_compatibility': 'Local SciPy lacks scalar_first; audit shim explicitly reorders scalar-last quaternions. Remote deployed source and libraries unchanged.', 'source_sha256': 'f1ddf4b5bf98c3c4d1be3c809208d4e0e820311e520bfddd33b3fbcd46f71745', 'cases': 3, 'input_dimensions': 22, 'simulator_action_dimensions': 23, 'hand16_preserved': True, 'position3_preserved': True, 'rotation_equivalence_max_abs_error': 3.3306690738754696e-16}


In [10]:
# Demonstrate the scalar-first / scalar-last error on the recorded mean action.
action=np.asarray(official["actions"]["mean"])
converted=probe._process_action(action)
wrong_rotation=Rotation.from_quat(converted[3:7]).as_matrix()
right_rotation=Rotation.from_rotvec(action[3:6]).as_matrix()
wrong_order_error=float(np.max(np.abs(wrong_rotation-right_rotation)))
assert wrong_order_error>1e-4
print("把标量在前四元数误当成标量在后时的旋转矩阵误差：",wrong_order_error)


把标量在前四元数误当成标量在后时的旋转矩阵误差： 1.3577478250034338


## 6. 下一步与影像的对应关系

归一化检查是数值检查，没有单独的“修复归一化前后”对照录像。已有策略回合在上一篇及动作时序篇播放。重跑时将每个回合绑定统计文件、相机键、动作布局和权重指纹，再看失败阶段；不要用一段成功录像替代变换校验。